# Demo de Inferență NUC-Net & Alpine
Acest notebook demonstrează cum se încărcă modelul NUC-Net (Segmentare Semantică) distilat, se încărcă ponderile pre-antrenate, se efectuează o inferență pe o scenă reală de ansamblu de puncte LiDAR și se rulează capl de clusterizare de instanțe Alpine.

In [12]:
import os
import sys
import zipfile

# Verificarea și extragerea libs.zip dacă directorul libs/ nu există
if not os.path.exists('libs'):
    if os.path.exists('libs.zip'):
        print("Se extrage libs.zip...")
        with zipfile.ZipFile('libs.zip', 'r') as zip_ref:
            zip_ref.extractall('.')
    else:
        print("WARNING: libs.zip nu a fost găsit!")

# Verificarea și extragerea data.zip dacă directorul data/ nu există
if not os.path.exists('data'):
    if os.path.exists('data.zip'):
        print("Se extrage data.zip...")
        with zipfile.ZipFile('data.zip', 'r') as zip_ref:
            zip_ref.extractall('.')
    else:
        print("WARNING: data.zip nu a fost găsit!")

# Adaugă librăriile locale în calea Python
sys.path.insert(0, os.path.abspath('libs/nucnet'))
sys.path.insert(0, os.path.abspath('libs/alpine'))

print("Modulele locale au fost încărcate cu succes!")


Se extrage libs.zip...
Se extrage data.zip...
Modulele locale au fost încărcate cu succes!


In [13]:
# Setarea mediului si a path-urilor
import os
import sys
import yaml
import time
import torch
import numpy as np
import plotly.graph_objects as go

from models.student import build_student
from dataloader.dataset_semantickitti import cylinder_dataset, collate_fn_BEV
from alpine import Alpine
from alpine_semantickitti import THING_CLASSES, BBOX_WEB, CLASSES

## Configurarea & Fișierele de Date
Se definesc path-urile către modelul pre-antrenat `.pt`, configurarea `.yaml` și scena de intrare `.bin` a ansamblului de puncte folosită în extragerea demo-ului.

In [15]:
INPUT_SCENE = "data/000000.bin"
PRETRAINED_MODEL = "data/student_ss_best.pt"
CONFIG_PATH = "data/distill_semantickitti.yaml"

print(f"Se foloseste scena de intrare: {INPUT_SCENE}")
print(f"Se foloseste modelul pre-antrenat: {PRETRAINED_MODEL}")
print(f"Se foloseste configuratia: {CONFIG_PATH}")

Se foloseste scena de intrare: data/000000.bin
Se foloseste modelul pre-antrenat: data/student_ss_best.pt
Se foloseste configuratia: data/distill_semantickitti.yaml


## Vizualizarea Ansamblului de Puncte Neprocesat

Mai întâi se definește o funcție ajutătoare pentru a vizualiza acele puncte.

In [16]:
# SemanticKITTI Color Map (BGR -> RGB) oficial
COLOR_MAP = {
    0: [0, 0, 0], 1: [100, 150, 245], 2: [100, 230, 245], 3: [30, 60, 150], 4: [80, 30, 180],
    5: [100, 80, 250], 6: [255, 30, 30], 7: [255, 40, 200], 8: [150, 30, 90], 9: [255, 0, 255],
    10: [255, 150, 255], 11: [75, 0, 75], 12: [175, 0, 75], 13: [255, 200, 0], 14: [255, 120, 50],
    15: [0, 175, 0], 16: [135, 60, 0], 17: [150, 240, 80], 18: [255, 240, 150], 19: [255, 0, 0],
}

# Functia care va fi folosita global pentru a genera graficul de vizualizare
def plot_point_cloud(points, colors, title, hover_texts=None, is_categorical=True):
    subsample = 10
    pts = points[::subsample]
    clrs = colors[::subsample] if not isinstance(colors, list) else [colors[i] for i in range(0, len(colors), subsample)]

    marker_dict = dict(size=2, color=clrs, opacity=0.8)
    if not is_categorical:
        marker_dict["colorscale"] = "Turbo"

    fig = go.Figure(data=[go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode="markers",
        marker=marker_dict,
        text=hover_texts[::subsample] if hover_texts else None,
        hoverinfo="text" if hover_texts else "none"
    )])
    # Facem marginile mai inguste pentru a maximiza area de vizualizare 3D
    fig.update_layout(
        title=title,
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, b=0, t=30)
    )
    fig.show()

Înainte de a aplica inferența, se pot vizualiza punctele LiDAR neprocesate, încărcate direct din fișierul `.bin`, colorate în funcție de înălțime (axa Z).

In [17]:
raw_pc = np.fromfile(INPUT_SCENE, dtype=np.float32).reshape((-1, 4))
xyz = raw_pc[:, :3]

# Randarea punctelor neprocesate, colorate dupa axa lor Z.
hover_z = [f"Z-inaltime: {z:.2f}m" for z in xyz[:, 2]]
plot_point_cloud(xyz, xyz[:, 2], "Ansamblul de puncte LiDAR Neprocesat (colorat după înălțime)", hover_texts=hover_z, is_categorical=False)

## Încărcarea Datelor & Voxelizarea
Se folosește voxelizarea oficială `cylinder_dataset` din `Cylinder3d` pe un wrapper personalizat pentru un dataset cu o singură scenă.

In [6]:
# Deoarece noi vrem sa procesam o singura scena, construim o clasa de tip torch dataset, pentru 
# o singura data neprocesata
class SinglePointCloudDataset(torch.utils.data.Dataset):
    '''
    Point Cloud class util to process single LiDAR scene.
    '''
    def __init__(self, bin_path):
        self.bin_path = bin_path
    def __len__(self):
        return 1
    def __getitem__(self, index):
        raw_data = np.fromfile(self.bin_path, dtype=np.float32).reshape((-1, 4))
        annotated_data = np.zeros((raw_data.shape[0], 1), dtype=np.int32)
        return (raw_data[:, :3], annotated_data.astype(np.uint8), raw_data[:, 3])

pt_dataset = SinglePointCloudDataset(INPUT_SCENE)

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

student_config = config["student_params"]
dataset_config = config["dataset_params"]

voxel_dataset = cylinder_dataset(
    pt_dataset,
    grid_size=student_config["output_shape"],
    fixed_volume_space=dataset_config["fixed_volume_space"],
    max_volume_space=dataset_config["max_volume_space"],
    min_volume_space=dataset_config["min_volume_space"],
    ignore_label=dataset_config["ignore_label"],
    return_test=False,
    num_scales=student_config["num_scales"]
)

batch = collate_fn_BEV([voxel_dataset[0]])
_, _, val_grid, _, val_pt_fea, val_grid_ms = batch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
pt_fea_ten = [torch.from_numpy(i).float().to(device) for i in val_pt_fea]
grid_ten = [torch.from_numpy(i).to(device) for i in val_grid]
grid_ms_ten = [torch.from_numpy(i).to(device) for i in val_grid_ms]

print(f"Norul de puncte a fost pregatit. Puncte grid: {len(grid_ten[0])}")

Norul de puncte a fost pregatit. Puncte grid: 123389


## Instanțierea Modelului & Pasul de Propagare
Se instanțiază modelul NUC-Net, se încărcă ponderile pre-antrenate și se trec datele prin model pentru a obține predicțiile de segmentare semantică.

In [7]:
model = build_student(student_config, device=str(device))

if os.path.exists(PRETRAINED_MODEL):
    state_dict = torch.load(PRETRAINED_MODEL, map_location=device)
    if "student_state_dict" in state_dict:
        state_dict = state_dict["student_state_dict"]
    model.load_state_dict(state_dict, strict=False)
    print("Modelul pre-antrenat a fost incarcat cu succes!")
else:
    print("Eroare: Modelul pre-antrenat nu a fost gasit!")

model.eval()
print("Rulare inferenta...")
start_time_nucnet = time.time()
with torch.no_grad():
    outputs = model(pt_fea_ten, grid_ten, 1, train_vox_ten_ms=grid_ms_ten)
    logits = outputs["dense_logits"] if isinstance(outputs, dict) else outputs
    predict_labels = torch.argmax(logits, dim=1).cpu().numpy()
end_time_nucnet = time.time()

per_point_pred = predict_labels[0, val_grid[0][:, 0], val_grid[0][:, 1], val_grid[0][:, 2]]
print(f"Inferenta semantica s-a incheiat! Dimensiunea etichetelor prezise: {per_point_pred.shape}")
print(f"Inferenta NUC-Net a durat: {end_time_nucnet - start_time_nucnet:.4f} secunde pe {device}")

[120 360  32]
[Student] Total params: 14.08M (trainable: 14.08M)
Modelul pre-antrenat a fost incarcat cu succes!
Rulare inferenta...
Inferenta semantica s-a incheiat! Dimensiunea etichetelor prezise: (123389,)
Inferenta NUC-Net a durat: 0.2275 secunde pe cuda:0


## Vizualizarea Segmentării Semantice
Aici sunt prezentate clasele semantice deduse de NUC-Net Distilat. Dacă se trece cu mouse-ul peste puncte, va apărea clasa lor semantică într-un format lizibil.

In [8]:
sem_colors = [f"rgb({COLOR_MAP[l][0]},{COLOR_MAP[l][1]},{COLOR_MAP[l][2]})" for l in per_point_pred]
sem_hover = [f"Clasa: {CLASSES.get(l, 'Unknown').title()}" for l in per_point_pred]
plot_point_cloud(xyz, sem_colors, "Rezultatele Segmentarii Semantice", hover_texts=sem_hover)

## Clusterizarea Instanțelor cu Alpine
Se invocă modulul de clusterizare **Alpine**.

In [9]:
# Instantierea modului oficial Alpine
alpine_model = Alpine(THING_CLASSES, BBOX_WEB, k=32, split=False, margin=1.3)

print("Rularea Alpine...")
start_time_alpine = time.time()
inst_pred = alpine_model.fit_predict(xyz[:, :2], per_point_pred)
end_time_alpine = time.time()
print("Clusterizarea Alpine a fost finalizata.")
print(f"Clusterizarea Alpine a durat: {end_time_alpine - start_time_alpine:.4f} secunde pe {device}")

Rularea Alpine...
Clusterizarea Alpine a fost finalizata.
Clusterizarea Alpine a durat: 0.0081 secunde pe cuda:0


Culorile semantice originale pentru toate clasele 'stuff' sunt menținute, și doar culorile pentru instanțele 'thing' sunt schimbate astfel încât fiecare instanță (de exemplu, mașini individuale) primește o nuanță aleatorie plus un identificator unic pentru a le separa vizual una de alta.

In [10]:
# Gruparea instantelor dupa clasa lor semantica pentru a genera variatii a culorii de baza 
from collections import defaultdict
sem_to_insts = defaultdict(set)
for sem, inst in zip(per_point_pred, inst_pred):
    if inst > 0:
        sem_to_insts[sem].add(inst)

inst_color_dict = {}
np.random.seed(42)
for sem, insts in sem_to_insts.items():
    base_color = COLOR_MAP[sem]
    for inst in insts:
        if len(insts) == 1:
            inst_color_dict[inst] = f"rgb({base_color[0]},{base_color[1]},{base_color[2]})"
        else:
            # Generarea variatiilor distincte pentru culoarea semantica de baza
            noise = np.random.randint(-50, 50, size=3)
            factor = np.random.uniform(0.6, 1.4)
            new_color = np.clip(np.array(base_color) * factor + noise, 0, 255).astype(int)
            inst_color_dict[inst] = f"rgb({new_color[0]},{new_color[1]},{new_color[2]})"

panoptic_colors = []
panoptic_hover = []

for sem, inst in zip(per_point_pred, inst_pred):
    class_name = CLASSES.get(sem, 'Unknown').title()
    if inst > 0:
        panoptic_colors.append(inst_color_dict[inst])
        panoptic_hover.append(f"Clasa: {class_name} | ID Instanta: {inst}")
    else:
        panoptic_colors.append(f"rgb({COLOR_MAP[sem][0]},{COLOR_MAP[sem][1]},{COLOR_MAP[sem][2]})")
        panoptic_hover.append(f"Clasa: {class_name}")

plot_point_cloud(xyz, panoptic_colors, "Rezultatul Final de Segmentare Panoptica Alpine", hover_texts=panoptic_hover)